# DHIS2 ↔ Sunbird RC Integration Adapter

This notebook syncs Water Facility records from DHIS2 to Sunbird RC.

## Sync Flow

```
DHIS2 (TEI with syncStatus=PENDING)
         │
         ▼
      Adapter (this notebook)
         │
         │ 1. Fetch pending records from DHIS2
         │ 2. Resolve org unit UIDs → names
         │ 3. Transform to Sunbird RC format
         │ 4. POST to Sunbird RC
         │ 5. Get osid & wfId
         │ 6. Update DHIS2 with IDs + syncStatus=SYNCED
         ▼
DHIS2 (Updated with osid, wfId, syncStatus=SYNCED)
```

## Prerequisites

- DHIS2 running at http://localhost:9090
- Sunbird RC running at http://localhost:8081
- Keycloak running at http://keycloak:8080
- Water Facility setup completed (dhis2-water-facility-setup.ipynb)

## 1. Setup & Configuration

In [1]:
import json
import os
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display, Markdown

# Load .env file (from project root)
def load_env():
    """Load environment variables from .env file."""
    env_path = Path("..") / ".env"
    if env_path.exists():
        with open(env_path) as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    key, value = line.split("=", 1)
                    os.environ.setdefault(key.strip(), value.strip())

load_env()

# Load config from JSON file (from adapter folder)
def load_config():
    """Load configuration from adapter/config.json."""
    config_path = Path("..") / "adapter" / "config.json"
    if not config_path.exists():
        raise Exception(f"Config file not found: {config_path}")
    with open(config_path) as f:
        return json.load(f)

CONFIG = load_config()

# Configuration (from environment variables)
DHIS2_URL = os.environ.get("DHIS2_URL", "http://localhost:9090/api")
DHIS2_AUTH = (
    os.environ.get("DHIS2_USERNAME", "admin"),
    os.environ.get("DHIS2_PASSWORD", "district"),
)

SUNBIRD_URL = os.environ.get("SUNBIRD_URL", "http://localhost:8081/api/v1")
KEYCLOAK_URL = os.environ.get(
    "KEYCLOAK_URL",
    "http://keycloak:8080/auth/realms/sunbird-rc/protocol/openid-connect/token"
)
CLIENT_ID = os.environ.get("SUNBIRD_CLIENT_ID", "demo-api")
CLIENT_SECRET = os.environ.get("SUNBIRD_CLIENT_SECRET", "")

# Build attribute codes list from config attributes
ATTR_CODES_LIST = [attr["code"] for attr in CONFIG["attributes"]]


# Build option mappings from config option_sets
def build_option_mappings():
    """Build option mappings for adapter from config option_sets."""
    mappings = {}
    # Create lookup: option_set_code -> attribute_code
    attr_to_option_set = {}
    for attr in CONFIG["attributes"]:
        if "optionSetCode" in attr:
            attr_to_option_set[attr["optionSetCode"]] = attr["code"]

    for option_set in CONFIG["option_sets"]:
        os_code = option_set["code"]
        attr_code = attr_to_option_set.get(os_code)
        if attr_code:
            mappings[attr_code] = {}
            for opt in option_set["options"]:
                full_code = f"{os_code}_{opt['code']}"
                mappings[attr_code][full_code] = opt["name"]
    return mappings


FIELD_MAPPING = CONFIG["field_mapping"]
OPTION_MAPPINGS = build_option_mappings()
ORG_UNIT_FIELDS = CONFIG["org_unit_fields"]
SYNC_STATUS = CONFIG["sync_status"]

# These will be populated dynamically
ROOT_OU_ID = None
PROGRAM_ID = None
ATTR_IDS = {}
ATTR_CODES = {}

# Cache for org unit names
ORG_UNIT_CACHE = {}

print(f"DHIS2 URL: {DHIS2_URL}")
print(f"Sunbird RC URL: {SUNBIRD_URL}")
print(f"Config loaded: {len(ATTR_CODES_LIST)} attributes")
print(f"Option mappings: {len(OPTION_MAPPINGS)} attribute mappings")

DHIS2 URL: http://localhost:9090/api
Sunbird RC URL: http://localhost:8081/api/v1
Config loaded: 17 attributes
Option mappings: 6 attribute mappings


## 2. Authentication

Test connections to both DHIS2 and Sunbird RC.

In [2]:
# Test DHIS2 connection
response = requests.get(
    f"{DHIS2_URL}/system/info",
    auth=DHIS2_AUTH,
    headers={"Accept": "application/json"}
)

if response.status_code == 200:
    info = response.json()
    print(f"DHIS2 Connected")
    print(f"  Version: {info.get('version')}")
    print(f"  Server Date: {info.get('serverDate')}")
else:
    print(f"DHIS2 connection failed: {response.status_code}")

DHIS2 Connected
  Version: 2.40.11
  Server Date: 2026-04-29T12:09:46.864


In [3]:
# Fetch DHIS2 IDs dynamically by code

def fetch_dhis2_ids():
    """Fetch DHIS2 IDs dynamically by code."""
    global ROOT_OU_ID, PROGRAM_ID, ATTR_IDS, ATTR_CODES

    headers = {"Accept": "application/json"}

    # Fetch root org unit (level 1)
    response = requests.get(
        f"{DHIS2_URL}/organisationUnits"
        f"?filter=level:eq:1&fields=id,name&paging=false",
        auth=DHIS2_AUTH,
        headers=headers,
    )
    if response.status_code != 200:
        raise Exception(
            f"Failed to fetch root org unit: {response.status_code}"
        )
    org_units = response.json().get("organisationUnits", [])
    if not org_units:
        raise Exception("No root organisation unit found")
    ROOT_OU_ID = org_units[0]["id"]

    # Fetch program by code (from config)
    prog_code = CONFIG["program"]["code"]
    response = requests.get(
        f"{DHIS2_URL}/programs"
        f"?filter=code:eq:{prog_code}&fields=id,name&paging=false",
        auth=DHIS2_AUTH,
        headers=headers,
    )
    if response.status_code != 200:
        raise Exception(f"Failed to fetch program: {response.status_code}")
    programs = response.json().get("programs", [])
    if not programs:
        raise Exception(
            f"Program {prog_code} not found. "
            "Run adapter/setup.py first."
        )
    PROGRAM_ID = programs[0]["id"]

    # Fetch tracked entity attributes by codes
    codes_param = ",".join(ATTR_CODES_LIST)
    response = requests.get(
        f"{DHIS2_URL}/trackedEntityAttributes"
        f"?filter=code:in:[{codes_param}]&fields=id,code&paging=false",
        auth=DHIS2_AUTH,
        headers=headers,
    )
    if response.status_code != 200:
        raise Exception(f"Failed to fetch attributes: {response.status_code}")
    attributes = response.json().get("trackedEntityAttributes", [])

    ATTR_IDS = {attr["code"]: attr["id"] for attr in attributes}
    ATTR_CODES = {v: k for k, v in ATTR_IDS.items()}

    # Verify all required attributes exist
    missing = [code for code in ATTR_CODES_LIST if code not in ATTR_IDS]
    if missing:
        raise Exception(
            f"Missing attributes: {missing}. Run adapter/setup.py first."
        )

    return True

# Fetch IDs
fetch_dhis2_ids()
print(f"Root OU: {ROOT_OU_ID}")
print(f"Program: {PROGRAM_ID}")
print(f"Attributes: {len(ATTR_IDS)} loaded")

Root OU: lAhVmHAInq2
Program: bNDnlEUnzBL
Attributes: 17 loaded


In [4]:
# Get Sunbird RC OAuth2 token from Keycloak

def get_sunbird_token():
    """Get OAuth2 access token from Keycloak."""
    data = {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "grant_type": "client_credentials",
    }
    response = requests.post(KEYCLOAK_URL, data=data)
    if response.status_code == 200:
        return response.json().get("access_token")
    raise Exception(f"Failed to get token: {response.status_code} {response.text}")

# Get token
ACCESS_TOKEN = get_sunbird_token()
AUTH_HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {ACCESS_TOKEN}"
}

print(f"Sunbird RC token acquired")
print(f"  Token (first 50 chars): {ACCESS_TOKEN[:50]}...")

Sunbird RC token acquired
  Token (first 50 chars): eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6IC...


In [5]:
# Test Sunbird RC connection
response = requests.get(
    f"{SUNBIRD_URL}/WaterFacility",
    headers=AUTH_HEADERS
)

if response.status_code == 200:
    facilities = response.json()
    print(f"Sunbird RC Connected")
    print(f"  Existing Water Facilities: {len(facilities)}")
else:
    print(f"Sunbird RC connection failed: {response.status_code}")

Sunbird RC Connected
  Existing Water Facilities: 2


## 3. Helper Functions

In [6]:
def dhis2_get(endpoint):
    """GET request to DHIS2 API."""
    url = f"{DHIS2_URL}/{endpoint}"
    response = requests.get(
        url, auth=DHIS2_AUTH, headers={"Accept": "application/json"}
    )
    return response


def dhis2_put(endpoint, data):
    """PUT request to DHIS2 API."""
    url = f"{DHIS2_URL}/{endpoint}"
    response = requests.put(
        url,
        auth=DHIS2_AUTH,
        headers={"Content-Type": "application/json", "Accept": "application/json"},
        json=data,
    )
    return response


def sunbird_post(endpoint, data):
    """POST request to Sunbird RC API."""
    url = f"{SUNBIRD_URL}/{endpoint}"
    response = requests.post(url, headers=AUTH_HEADERS, json=data)
    return response


def sunbird_get(endpoint):
    """GET request to Sunbird RC API."""
    url = f"{SUNBIRD_URL}/{endpoint}"
    response = requests.get(
        url, headers={"Accept": "application/json", "Authorization": AUTH_HEADERS["Authorization"]}
    )
    return response


def resolve_org_unit(uid):
    """Convert org unit UID to name."""
    if not uid:
        return None
    if uid in ORG_UNIT_CACHE:
        return ORG_UNIT_CACHE[uid]

    response = dhis2_get(f"organisationUnits/{uid}?fields=name")
    if response.status_code == 200:
        name = response.json().get("name")
        ORG_UNIT_CACHE[uid] = name
        return name
    return None


def get_attribute_value(tei, attr_code):
    """Extract attribute value from TEI by attribute code."""
    attr_id = ATTR_IDS.get(attr_code)
    if not attr_id:
        return None
    for attr in tei.get("attributes", []):
        if attr.get("attribute") == attr_id:
            return attr.get("value")
    return None


def option_code_to_name(code, attr_code):
    """Convert DHIS2 option code to display name using config."""
    if not code:
        return None
    option_map = OPTION_MAPPINGS.get(attr_code, {})
    return option_map.get(code, code)


print("Helper functions defined")

Helper functions defined


## 4. Fetch Pending Records from DHIS2

Query TEIs where syncStatus = PENDING.

In [7]:
def fetch_pending_teis():
    """Fetch TEIs with syncStatus=PENDING from DHIS2."""
    sync_status_id = ATTR_IDS["SYNC_STATUS_ATTR"]
    pending_status = SYNC_STATUS["pending"]
    endpoint = (
        f"trackedEntityInstances"
        f"?ou={ROOT_OU_ID}"
        f"&ouMode=DESCENDANTS"
        f"&program={PROGRAM_ID}"
        f"&filter={sync_status_id}:eq:{pending_status}"
        f"&fields=*"
        f"&paging=false"
    )
    response = dhis2_get(endpoint)
    if response.status_code != 200:
        raise Exception(f"Failed to fetch TEIs: {response.status_code} {response.text}")

    data = response.json()
    teis = data.get("trackedEntityInstances", [])
    return teis

# Fetch pending records
pending_teis = fetch_pending_teis()
print(f"Found {len(pending_teis)} pending records")

# Display as table
if pending_teis:
    rows = []
    for tei in pending_teis:
        rows.append({
            "TEI ID": tei.get("trackedEntityInstance"),
            "Geo Code": get_attribute_value(tei, "GEO_CODE"),
            "County": resolve_org_unit(get_attribute_value(tei, "COUNTY")),
            "District": resolve_org_unit(get_attribute_value(tei, "DISTRICT")),
            "Community": resolve_org_unit(get_attribute_value(tei, "COMMUNITY")),
            "Water Point Type": option_code_to_name(
                get_attribute_value(tei, "WATER_POINT_TYPE_ATTR"), "WATER_POINT_TYPE_ATTR"
            ),
        })
    df = pd.DataFrame(rows)
    display(df)
else:
    print("No pending records to sync.")

Found 0 pending records
No pending records to sync.


## 5. Transform DHIS2 → Sunbird RC Format

Transform TEI attributes to Sunbird RC WaterFacility schema.

In [8]:
def transform_tei_to_sunbird(tei):
    """Transform DHIS2 TEI to Sunbird RC WaterFacility format."""
    # Get basic fields
    geo_code = get_attribute_value(tei, "GEO_CODE")

    # Resolve org unit UIDs to names
    county_uid = get_attribute_value(tei, "COUNTY")
    district_uid = get_attribute_value(tei, "DISTRICT")
    community_uid = get_attribute_value(tei, "COMMUNITY")

    county_name = resolve_org_unit(county_uid)
    district_name = resolve_org_unit(district_uid)
    community_name = resolve_org_unit(community_uid)

    # Get coordinates from TEI geometry
    geometry = tei.get("geometry")
    coordinates = None
    if geometry and geometry.get("type") == "Point":
        coords = geometry.get("coordinates", [])
        if len(coords) >= 2:
            # DHIS2 stores as [lon, lat]
            coordinates = {"lat": coords[1], "lon": coords[0]}

    # Map option codes to display names (using config)
    water_point_type = option_code_to_name(
        get_attribute_value(tei, "WATER_POINT_TYPE_ATTR"), "WATER_POINT_TYPE_ATTR"
    )
    extraction_type = option_code_to_name(
        get_attribute_value(tei, "EXTRACTION_TYPE_ATTR"), "EXTRACTION_TYPE_ATTR"
    )
    pump_type = option_code_to_name(
        get_attribute_value(tei, "PUMP_TYPE_ATTR"), "PUMP_TYPE_ATTR"
    )
    installer = option_code_to_name(
        get_attribute_value(tei, "INSTALLER"), "INSTALLER"
    )
    owner = option_code_to_name(get_attribute_value(tei, "OWNER"), "OWNER")

    # Get other fields
    num_taps = get_attribute_value(tei, "NUM_TAPS")
    has_depth_info = get_attribute_value(tei, "HAS_DEPTH_INFO")
    depth_metres = get_attribute_value(tei, "DEPTH_METRES")
    funder = get_attribute_value(tei, "FUNDER")
    photo_url = get_attribute_value(tei, "PHOTO_URL")

    # Build Sunbird RC payload
    sunbird_data = {
        "geoCode": geo_code,
        "waterPointType": water_point_type,
        "location": {
            "county": county_name,
            "district": district_name,
            "community": community_name,
        },
    }

    # Add coordinates if available
    if coordinates:
        sunbird_data["location"]["coordinates"] = coordinates

    # Add optional fields if present
    if extraction_type:
        sunbird_data["extractionType"] = extraction_type
    if pump_type:
        sunbird_data["pumpType"] = pump_type
    if installer:
        sunbird_data["installer"] = installer
    if owner:
        sunbird_data["owner"] = owner
    if num_taps:
        sunbird_data["numTaps"] = int(num_taps)
    if has_depth_info is not None:
        sunbird_data["hasDepthInfo"] = has_depth_info == "true"
    if depth_metres:
        sunbird_data["depthMetres"] = float(depth_metres)
    if funder:
        sunbird_data["funder"] = funder
    if photo_url:
        sunbird_data["photoUrl"] = photo_url

    return sunbird_data


# Preview transformation for first pending TEI
if pending_teis:
    sample_tei = pending_teis[0]
    sample_sunbird = transform_tei_to_sunbird(sample_tei)
    print("Sample transformation (DHIS2 → Sunbird RC):")
    print(json.dumps(sample_sunbird, indent=2))

## 6. Send to Sunbird RC & Update DHIS2

In [9]:
def update_tei_with_ids(tei_id, osid, wf_id, org_unit_id):
    """Update DHIS2 TEI with Sunbird RC generated IDs."""
    attributes = [
        {"attribute": ATTR_IDS["SUNBIRD_OSID"], "value": osid},
        {"attribute": ATTR_IDS["WF_ID"], "value": wf_id},
        {"attribute": ATTR_IDS["SYNC_STATUS_ATTR"], "value": SYNC_STATUS["synced"]},
    ]

    payload = {"orgUnit": org_unit_id, "attributes": attributes}

    response = dhis2_put(f"trackedEntityInstances/{tei_id}", payload)
    return response.status_code in [200, 204]


def set_tei_failed(tei_id, org_unit_id):
    """Set TEI syncStatus to FAILED."""
    attributes = [
        {"attribute": ATTR_IDS["SYNC_STATUS_ATTR"], "value": SYNC_STATUS["failed"]},
    ]

    payload = {"orgUnit": org_unit_id, "attributes": attributes}
    dhis2_put(f"trackedEntityInstances/{tei_id}", payload)


def sync_single_tei(tei):
    """Sync a single TEI to Sunbird RC."""
    tei_id = tei.get("trackedEntityInstance")
    org_unit_id = tei.get("orgUnit")
    geo_code = get_attribute_value(tei, "GEO_CODE")

    # Transform to Sunbird RC format
    sunbird_data = transform_tei_to_sunbird(tei)

    # POST to Sunbird RC
    response = sunbird_post("WaterFacility", sunbird_data)

    if response.status_code not in [200, 201]:
        set_tei_failed(tei_id, org_unit_id)
        return {
            "tei_id": tei_id,
            "geo_code": geo_code,
            "status": "FAILED",
            "error": f"Sunbird create failed: {response.status_code}",
        }

    # Extract osid from response
    result = response.json()
    osid = result.get("result", {}).get("WaterFacility", {}).get("osid")

    if not osid:
        set_tei_failed(tei_id, org_unit_id)
        return {
            "tei_id": tei_id,
            "geo_code": geo_code,
            "status": "FAILED",
            "error": "No osid in response",
        }

    # GET the created record to fetch wfId
    get_response = sunbird_get(f"WaterFacility/{osid}")

    if get_response.status_code != 200:
        set_tei_failed(tei_id, org_unit_id)
        return {
            "tei_id": tei_id,
            "geo_code": geo_code,
            "status": "FAILED",
            "error": f"Could not fetch record: {get_response.status_code}",
        }

    facility = get_response.json()
    wf_id = facility.get("wfId", "")

    # Update DHIS2 with IDs
    if update_tei_with_ids(tei_id, osid, wf_id, org_unit_id):
        return {
            "tei_id": tei_id,
            "geo_code": geo_code,
            "status": "SUCCESS",
            "osid": osid,
            "wf_id": wf_id,
        }
    else:
        return {
            "tei_id": tei_id,
            "geo_code": geo_code,
            "status": "FAILED",
            "error": "Could not update DHIS2",
        }


print("Sync functions defined")

Sync functions defined


## 7. Run Sync

In [10]:
def run_sync():
    """Run full sync of pending TEIs."""
    print("=" * 60)
    print("DHIS2 → Sunbird RC Sync")
    print(f"Started: {datetime.now().isoformat()}")
    print("=" * 60)

    # Refresh token
    global ACCESS_TOKEN, AUTH_HEADERS
    ACCESS_TOKEN = get_sunbird_token()
    AUTH_HEADERS = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {ACCESS_TOKEN}"
    }
    print("\n1. Token refreshed")

    # Fetch pending TEIs
    print("\n2. Fetching pending TEIs...")
    teis = fetch_pending_teis()
    print(f"   Found {len(teis)} pending records")

    if not teis:
        print("\nNo pending records to sync.")
        return []

    # Sync each TEI
    print("\n3. Syncing records...")
    results = []
    for i, tei in enumerate(teis, 1):
        geo_code = get_attribute_value(tei, "GEO_CODE")
        print(f"   [{i}/{len(teis)}] {geo_code}...", end=" ")
        result = sync_single_tei(tei)
        results.append(result)
        print(result["status"])

    # Summary
    success_count = len([r for r in results if r["status"] == "SUCCESS"])
    fail_count = len([r for r in results if r["status"] == "FAILED"])

    print("\n" + "=" * 60)
    print("Sync Summary")
    print("=" * 60)
    print(f"  Total processed: {len(results)}")
    print(f"  Successful: {success_count}")
    print(f"  Failed: {fail_count}")
    print(f"Finished: {datetime.now().isoformat()}")

    return results

In [11]:
# Run the sync
sync_results = run_sync()

DHIS2 → Sunbird RC Sync
Started: 2026-04-29T19:09:47.254579

1. Token refreshed

2. Fetching pending TEIs...
   Found 0 pending records

No pending records to sync.


## 8. Sync Results

In [12]:
# Display sync results as table
if sync_results:
    df = pd.DataFrame(sync_results)
    display(df)
else:
    print("No sync results to display.")

No sync results to display.


## 9. Manual Sync (Interactive)

Functions to sync individual TEIs or retry failed records.

In [13]:
def sync_tei_by_id(tei_id):
    """Sync a single TEI by ID."""
    # Refresh token
    global ACCESS_TOKEN, AUTH_HEADERS
    ACCESS_TOKEN = get_sunbird_token()
    AUTH_HEADERS = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {ACCESS_TOKEN}"
    }

    # Fetch the TEI
    response = dhis2_get(f"trackedEntityInstances/{tei_id}?fields=*")
    if response.status_code != 200:
        print(f"Failed to fetch TEI: {response.status_code}")
        return None

    tei = response.json()
    result = sync_single_tei(tei)

    if result["status"] == "SUCCESS":
        print(f"SUCCESS: {result['geo_code']}")
        print(f"  osid: {result['osid']}")
        print(f"  wfId: {result['wf_id']}")
    else:
        print(f"FAILED: {result['geo_code']}")
        print(f"  Error: {result.get('error')}")

    return result


def retry_failed_records():
    """Retry syncing records with syncStatus=FAILED."""
    sync_status_id = ATTR_IDS["SYNC_STATUS_ATTR"]
    failed_status = SYNC_STATUS["failed"]
    endpoint = (
        f"trackedEntityInstances"
        f"?ou={ROOT_OU_ID}"
        f"&ouMode=DESCENDANTS"
        f"&program={PROGRAM_ID}"
        f"&filter={sync_status_id}:eq:{failed_status}"
        f"&fields=*"
        f"&paging=false"
    )
    response = dhis2_get(endpoint)
    if response.status_code != 200:
        print(f"Failed to fetch failed TEIs: {response.status_code}")
        return []

    teis = response.json().get("trackedEntityInstances", [])
    print(f"Found {len(teis)} failed records to retry")

    if not teis:
        return []

    # Refresh token
    global ACCESS_TOKEN, AUTH_HEADERS
    ACCESS_TOKEN = get_sunbird_token()
    AUTH_HEADERS = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {ACCESS_TOKEN}"
    }

    results = []
    for tei in teis:
        result = sync_single_tei(tei)
        results.append(result)
        print(f"  {result['geo_code']}: {result['status']}")

    return results


print("Manual sync functions available:")
print("  sync_tei_by_id('TEI_ID') - Sync single TEI")
print("  retry_failed_records() - Retry all failed records")

Manual sync functions available:
  sync_tei_by_id('TEI_ID') - Sync single TEI
  retry_failed_records() - Retry all failed records


In [14]:
# Example: Sync a specific TEI
# sync_tei_by_id("YOUR_TEI_ID_HERE")

In [15]:
# Example: Retry failed records
# retry_failed_records()

## 10. Verification

Verify sync status and compare counts between systems.

In [16]:
# Count records by sync status in DHIS2
sync_status_id = ATTR_IDS["SYNC_STATUS_ATTR"]
statuses = [SYNC_STATUS["pending"], SYNC_STATUS["synced"], SYNC_STATUS["failed"]]

print("DHIS2 Water Facility Records:")
print("-" * 40)

for status in statuses:
    endpoint = (
        f"trackedEntityInstances"
        f"?ou={ROOT_OU_ID}"
        f"&ouMode=DESCENDANTS"
        f"&program={PROGRAM_ID}"
        f"&filter={sync_status_id}:eq:{status}"
        f"&paging=false"
        f"&fields=trackedEntityInstance"
    )
    response = dhis2_get(endpoint)
    if response.status_code == 200:
        count = len(response.json().get("trackedEntityInstances", []))
        status_name = status.replace("SYNC_STATUS_", "")
        print(f"  {status_name}: {count}")

DHIS2 Water Facility Records:
----------------------------------------
  PENDING: 0
  SYNCED: 9
  FAILED: 0


In [17]:
# Count records in Sunbird RC
response = sunbird_get("WaterFacility")
if response.status_code == 200:
    facilities = response.json()
    print(f"\nSunbird RC Water Facilities: {len(facilities)}")
else:
    print(f"Failed to fetch Sunbird RC records: {response.status_code}")


Sunbird RC Water Facilities: 2


In [18]:
# List synced records with their IDs
sync_status_id = ATTR_IDS["SYNC_STATUS_ATTR"]
synced_status = SYNC_STATUS["synced"]
endpoint = (
    f"trackedEntityInstances"
    f"?ou={ROOT_OU_ID}"
    f"&ouMode=DESCENDANTS"
    f"&program={PROGRAM_ID}"
    f"&filter={sync_status_id}:eq:{synced_status}"
    f"&paging=false"
    f"&fields=*"
)
response = dhis2_get(endpoint)

if response.status_code == 200:
    synced_teis = response.json().get("trackedEntityInstances", [])
    if synced_teis:
        print(f"\nSynced Records ({len(synced_teis)}):")
        print("-" * 80)
        rows = []
        for tei in synced_teis:
            rows.append({
                "TEI ID": tei.get("trackedEntityInstance"),
                "Geo Code": get_attribute_value(tei, "GEO_CODE"),
                "osid": get_attribute_value(tei, "SUNBIRD_OSID"),
                "wfId": get_attribute_value(tei, "WF_ID"),
            })
        df = pd.DataFrame(rows)
        display(df)
    else:
        print("No synced records found.")


Synced Records (9):
--------------------------------------------------------------------------------


,TEI ID,Geo Code,osid,wfId
0,VNWD1qiNltH,TEST1777345880,1-d363c332-a4f9-4036-89e7-5ecb22bab47f,WF-MON-GRE-TWB-243C71
1,iNqjcuX4NMr,WF1777459392,1-02642a98-cd20-489f-a087-d3f5ba3959dc,WF-MON-GRE-PDW-28A90A
2,diZCFPVBHjO,WF1777459873,1-d625b729-ee45-4ce3-914b-82d2bf64ce25,WF-MON-GRE-PDW-4C1E75
3,DZV8maNDh4y,WF1777462297,1-9cda9ce8-d795-4c30-bde9-b870fbb0e00d,WF-MON-GRE-PDW-A6970B
4,ktOGGrAKepd,WF1777463683733,1-a8920033-8ebb-4ad5-895e-725aa4b6ab8a,WF-NIM-SAN-SSD-674F1E
5,DawgdYSswTO,WF1777463770144,1-9bd8ff06-6f13-46b9-b6af-c393dd58a187,WF-MON-GRE-PWD-B86D56
6,RhMnKehnudy,WF1777463770262,1-00234916-677b-43f4-9516-8b4148911dd2,WF-MON-GRE-US-0B2391
7,udgWu47cpHU,WF1777463770327,1-8c985f3f-61e5-4ae6-b2a8-a4f31ec9d64a,WF-MON-GRE-RWH-591E10
8,b7i00RZEY1m,NIMBA1777463826,1-b8183ce9-9b16-4517-aa20-16354e61f149,WF-NIM-SAN-SSD-A54D66
